# Week 2 Day 3 — Cross Validation and Data Leakage

## What is data leakage:
When your model sees information during training that it 
shouldn't have — making it look better than it really is.

## Two types demonstrated:
1. Scaler leakage — fitting StandardScaler on all data
   before CV. Difference small on clean datasets.

2. Feature selection leakage — fitting SelectKBest on 
   all data before CV. Difference = 1.6% inflation.
   In production this means your model underperforms vs expectations.

## The rule:
Any step that calls fit() must go INSIDE the Pipeline.
Pipeline guarantees no leakage — every step fits only
on the train fold during cross validation.

## Leakage demonstration results:
- Leaky accuracy   : 0.8460 (fake)
- Correct accuracy : 0.8300 (real)
- Difference       : 0.016  (1.6% inflation)

# Demonstrate data leakage with real numbers:

In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

housing = fetch_california_housing()
X = housing.data
y = housing.target

# ── WRONG WAY — leaky ────────────────────────────────────────
scaler_leaky = StandardScaler()
X_leaky      = scaler_leaky.fit_transform(X)  # fits on ALL data

leaky_scores = cross_val_score(
    Ridge(alpha=1.0), X_leaky, y,
    cv=5, scoring='neg_root_mean_squared_error')

# ── CORRECT WAY — pipeline ───────────────────────────────────
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge',  Ridge(alpha=1.0))
])

correct_scores = cross_val_score(
    pipe, X, y,
    cv=5, scoring='neg_root_mean_squared_error')

print("=" * 50)
print("   DATA LEAKAGE DEMONSTRATION")
print("=" * 50)
print(f"  Leaky RMSE   : {-leaky_scores.mean():.4f}  ← fake good score")
print(f"  Correct RMSE : {-correct_scores.mean():.4f}  ← real score")
print(f"  Difference   : {(-leaky_scores.mean() - -correct_scores.mean()):.4f}")
print("=" * 50)

   DATA LEAKAGE DEMONSTRATION
  Leaky RMSE   : 0.7459  ← fake good score
  Correct RMSE : 0.7459  ← real score
  Difference   : -0.0000


# Another Dataset:

In [2]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

# create dataset with high variance features — leakage shows here
np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=50, 
                            n_informative=5, random_state=42)

# ── WRONG WAY — leaky ────────────────────────────────────────
scaler_leaky = StandardScaler()
X_leaky      = scaler_leaky.fit_transform(X)  # fits on ALL data

leaky_scores = cross_val_score(
    LogisticRegression(), X_leaky, y,
    cv=5, scoring='accuracy')

# ── CORRECT WAY — pipeline ───────────────────────────────────
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression())
])

correct_scores = cross_val_score(
    pipe, X, y,
    cv=5, scoring='accuracy')

print("=" * 50)
print("   DATA LEAKAGE DEMONSTRATION")
print("=" * 50)
print(f"  Leaky accuracy   : {leaky_scores.mean():.4f}  ← fake good score")
print(f"  Correct accuracy : {correct_scores.mean():.4f}  ← real score")
print(f"  Difference       : {(leaky_scores.mean() - correct_scores.mean()):.4f}")
print("=" * 50)

   DATA LEAKAGE DEMONSTRATION
  Leaky accuracy   : 0.7900  ← fake good score
  Correct accuracy : 0.7900  ← real score
  Difference       : 0.0000


# Another Dataset

In [3]:
from sklearn.feature_selection import SelectKBest, f_classif

# ── WRONG WAY — select features BEFORE cross validation ──────
selector_leaky = SelectKBest(f_classif, k=10)
X_leaky = selector_leaky.fit_transform(X, y)  # sees ALL data including val/test

leaky_scores = cross_val_score(
    LogisticRegression(), X_leaky, y,
    cv=5, scoring='accuracy')

# ── CORRECT WAY — feature selection INSIDE pipeline ──────────
pipe = Pipeline([
    ('selector', SelectKBest(f_classif, k=10)),
    ('scaler',   StandardScaler()),
    ('model',    LogisticRegression())
])

correct_scores = cross_val_score(
    pipe, X, y,
    cv=5, scoring='accuracy')

print("=" * 55)
print("   FEATURE SELECTION LEAKAGE DEMONSTRATION")
print("=" * 55)
print(f"  Leaky accuracy   : {leaky_scores.mean():.4f}  ← inflated fake score")
print(f"  Correct accuracy : {correct_scores.mean():.4f}  ← real score")
print(f"  Difference       : {(leaky_scores.mean() - correct_scores.mean()):.4f}")
print("=" * 55)

   FEATURE SELECTION LEAKAGE DEMONSTRATION
  Leaky accuracy   : 0.8460  ← inflated fake score
  Correct accuracy : 0.8300  ← real score
  Difference       : 0.0160
